# Query Decomposition [Step 3 - Breaking Complex Queries into Sub-Queries]

> **MLCourse - Agentic AI - Agentic RAG**

Complex questions often cannot be answered with a single retrieval. Query
decomposition breaks a multi-part question into focused sub-queries, each
targeting a specific aspect. This notebook builds a chain that decomposes
a complex query, retrieves for each sub-query, and synthesizes a combined
answer.

In [1]:
import os
import warnings
warnings.filterwarnings("ignore")
from dotenv import load_dotenv
load_dotenv()

False

In [2]:
api_key = os.environ.get("OPENAI_API_KEY", "")
if api_key:
    print("[GREEN] API key found")
else:
    print("[GREEN] No API key needed -- using local ChatOllama")

[GREEN] No API key needed -- using local ChatOllama


### 1. Load and Chunk the Document


In [ ]:
from langchain_text_splitters import CharacterTextSplitter

text_path = r"D:\projects\python\MLCourse\03_agentic_ai\data\alice.txt"
with open(text_path, "r", encoding="utf-8") as f:
    raw_text = f.read()

splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.split_text(raw_text)
print(f"Loaded alice.txt: {len(raw_text)} chars -> {len(chunks)} chunks")


### 2. Build the Vector Store


In [ ]:
from langchain_ollama import OllamaEmbeddings
from langchain_chroma import Chroma

embeddings = OllamaEmbeddings(model="nomic-embed-text")
vectorstore = Chroma.from_texts(chunks, embeddings, collection_name="alice_decomp")
print(f"Vector store built with {vectorstore._collection.count()} vectors")


### 3. Initialize the LLM


In [ ]:
from langchain_ollama import ChatOllama

llm = ChatOllama(model="llama3.1:8b", temperature=0)
print("LLM initialized:", llm.model)


### 4. Build the Decomposition Chain


In [ ]:
# The decomposition prompt asks the LLM to break a complex query into
# focused sub-queries. Each sub-query targets one specific aspect.

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

decompose_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a query decomposition expert. Break complex questions into "
     "simpler, self-contained sub-queries that can each be answered independently.\n\n"
     "RULES:\n"
     "- Each sub-query must be specific and self-contained\n"
     "- Preserve the original intent of each part\n"
     "- Order sub-queries logically (general first, then specific)\n"
     "- Do NOT repeat sub-queries that are nearly identical\n"
     "- Return ONLY the sub-queries, one per line, numbered 1, 2, 3, etc."),
    ("user", "{query}")
])

decompose_chain = decompose_prompt | llm | StrOutputParser()
print("Decomposition chain built.")


### 5. Test Decomposition on a Complex Query


In [ ]:
complex_query = (
    "What are the main themes in Alice in Wonderland "
    "and how do they relate to Victorian society?"
)

print("=" * 60)
print(f"Complex query: {complex_query}")
print("=" * 60)

result = decompose_chain.invoke({"query": complex_query})
print("\nDecomposed sub-queries:")
print(result)


### 6. Parse Sub-Queries into a List


In [ ]:
import re

def parse_sub_queries(text: str) -> list:
    """Extract numbered sub-queries from LLM output."""
    lines = text.strip().split("\n")
    queries = []
    for line in lines:
        # Remove numbering: "1. ", "1) ", etc.
        cleaned = re.sub(r"^\d+[\.\)]\s*", "", line.strip())
        if cleaned and len(cleaned) > 5:
            queries.append(cleaned)
    return queries

sub_queries = parse_sub_queries(result)
print(f"Parsed {len(sub_queries)} sub-queries:")
for i, sq in enumerate(sub_queries, 1):
    print(f"  {i}. {sq}")


### 7. Build the Retrieval Function


In [ ]:
# Simple similarity search for each sub-query.

def retrieve_for_query(query: str, k: int = 3) -> str:
    """Retrieve relevant documents for a single query."""
    docs = vectorstore.similarity_search(query, k=k)
    context = "\n\n".join([d.page_content for d in docs])
    return context


### 8. Build the Sub-Query Answering Chain


In [ ]:
# For each sub-query, retrieve and generate a focused answer.

sub_answer_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "Answer the question using ONLY the provided context. "
     "Be specific and concise. If the context is insufficient, say so."),
    ("user", "Context:\n{context}\n\nQuestion: {query}")
])

sub_answer_chain = sub_answer_prompt | llm | StrOutputParser()
print("Sub-answer chain built.")


### 9. Build the Synthesis Chain


In [ ]:
# Combines all sub-answers into a coherent response.

synthesis_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a synthesis expert. Combine the following sub-answers into a "
     "single, coherent response to the original complex question.\n\n"
     "RULES:\n"
     "- Maintain logical flow between topics\n"
     "- Remove redundancy across sub-answers\n"
     "- Ensure the final answer directly addresses the original question\n"
     "- Be comprehensive but concise"),
    ("user",
     "Original question: {original_query}\n\n"
     "Sub-answers:\n{sub_answers}")
])

synthesis_chain = synthesis_prompt | llm | StrOutputParser()
print("Synthesis chain built.")


### 10. Build the Full Decomposition Pipeline


In [ ]:
def decompose_and_answer(query: str) -> dict:
    """Full pipeline: decompose -> retrieve -> answer each -> synthesize."""
    print(f"\n{'='*60}")
    print(f"Processing: {query}")
    print(f"{'='*60}")

    # Step 1: Decompose
    print("\n[STEP 1] Decomposing query...")
    raw_decomp = decompose_chain.invoke({"query": query})
    sub_queries = parse_sub_queries(raw_decomp)
    print(f"  Got {len(sub_queries)} sub-queries")

    # Step 2: Retrieve and answer each sub-query
    print("\n[STEP 2] Retrieving and answering sub-queries...")
    sub_answers = []
    for i, sq in enumerate(sub_queries, 1):
        print(f"\n  Sub-query {i}: {sq[:80]}...")
        context = retrieve_for_query(sq)
        answer = sub_answer_chain.invoke({"context": context, "query": sq})
        sub_answers.append(f"Sub-query {i}: {sq}\nAnswer: {answer}")
        print(f"  Answer {i}: {answer[:100]}...")

    # Step 3: Synthesize
    print("\n[STEP 3] Synthesizing final answer...")
    combined = "\n\n".join(sub_answers)
    final = synthesis_chain.invoke({
        "original_query": query,
        "sub_answers": combined
    })
    print(f"  Final answer: {final[:150]}...")

    return {
        "original_query": query,
        "sub_queries": sub_queries,
        "sub_answers": sub_answers,
        "final_answer": final
    }


### 11. Run the Full Pipeline


In [ ]:
result = decompose_and_answer(
    "What are the main themes in Alice in Wonderland "
    "and how do they relate to Victorian society?"
)

print("\n" + "=" * 60)
print("FINAL ANSWER")
print("=" * 60)
print(result["final_answer"])


### 12. Test with a Different Complex Query


In [ ]:
result2 = decompose_and_answer(
    "Compare how Alice treats animals versus how humans treat her, "
    "and identify what this says about the book's message."
)

print("\n" + "=" * 60)
print("FINAL ANSWER")
print("=" * 60)
print(result2["final_answer"])


### 13. Test with a Three-Part Query


In [ ]:
result3 = decompose_and_answer(
    "Describe the Mad Hatter's tea party, explain why it is considered "
    "nonsensical, and compare it to real Victorian social customs."
)

print("\n" + "=" * 60)
print("FINAL ANSWER")
print("=" * 60)
print(result3["final_answer"])


### 14. Visualize the Pipeline as a Graph


In [ ]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict

class DecompState(TypedDict):
    query: str
    sub_queries: list
    sub_answers: list
    final_answer: str

print("=== Decomposition Pipeline Structure ===")
print("  START -> decompose")
print("           |")
print("           v")
print("  For each sub-query:")
print("    retrieve -> sub_answer")
print("           |")
print("           v")
print("  synthesize -> END")
print()
print("This is a sequential pipeline, not a branching graph.")
print("Each sub-query is handled independently, then combined.")


### Summary
